In [2]:

import pandas as pd, numpy as np, json, math, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore")

PANEL_PATH = "mnt/data/panel_sales_orders_stock.csv"
CFG_PATH   = "mnt/data/sku_config.csv"
OUT_DIR    = Path("mnt/data/v7_per_sku_outputs")
(OUT_DIR/"forecasts").mkdir(parents=True, exist_ok=True)
(OUT_DIR/"debug").mkdir(parents=True, exist_ok=True)

rng_global = np.random.default_rng(42)

def ensure_ms_freq(df):
    d = df.copy().sort_values("ds")
    d["ds"] = pd.to_datetime(d["ds"]).dt.to_period("M").dt.to_timestamp(how="start")
    d = d.drop_duplicates(["ds"])
    rng = pd.date_range(d["ds"].min(), d["ds"].max(), freq="MS")
    dd = pd.DataFrame({"ds": rng}).merge(d, on="ds", how="left")
    for c in ["y","orders"]:
        if c in dd.columns:
            dd[c] = pd.to_numeric(dd[c], errors="coerce").fillna(0.0).clip(lower=0)
    if "stock" in dd.columns:
        dd["stock"] = pd.to_numeric(dd["stock"], errors="coerce").ffill().fillna(0.0).clip(lower=0)
    return dd

def add_cal_and_lags(df):
    d = df.copy().sort_values("ds")
    d["month"] = d["ds"].dt.month
    d["year"]  = d["ds"].dt.year
    d["orders_ratio"] = d["orders"] / d["stock"].replace(0,np.nan)
    d["orders_ratio"] = d["orders_ratio"].replace([np.inf,-np.inf], np.nan).fillna(0.0)
    for c in ["y","orders","stock"]:
        d[f"{c}_lag1"] = d[c].shift(1)
        d[f"{c}_lag3"] = d[c].shift(3)
    lag_cols = [c for c in d.columns if c.endswith("_lag1") or c.endswith("_lag3")]
    d[lag_cols] = d[lag_cols].fillna(0.0)
    return d

def seasonal_naive(series, steps):
    s = series.values
    if len(s) >= 12:
        patt = s[-12:]
        reps = int(np.ceil(steps/12))
        fc = np.tile(patt, reps)[:steps]
    else:
        fc = np.repeat(s[-1] if len(s)>0 else 0.0, steps)
    return fc

def moving_avg(series, steps, window=3):
    base = series.rolling(window=window, min_periods=1).mean()
    last = float(base.iloc[-1]) if len(base)>0 else 0.0
    return np.repeat(max(0.0,last), steps)

def last_value(series, steps):
    last = float(series.iloc[-1]) if len(series)>0 else 0.0
    return np.repeat(max(0.0,last), steps)

def inverse_mae_weights(y_true, preds):
    maes = {}
    for k, yhat in preds.items():
        if len(y_true) == len(yhat) and len(y_true)>0:
            err = np.abs(y_true - yhat).mean()
        else:
            err = np.inf
        maes[k] = err
    inv = {k: (0.0 if (not np.isfinite(v) or v<=0) else 1.0/v) for k,v in maes.items()}
    s = sum(inv.values())
    if s == 0:
        K = len(preds) if len(preds)>0 else 1
        return {k: 1.0/K for k in preds}
    return {k: v/s for k,v in inv.items()}

def exog_forecast_inverse_ensemble(df_hist, col, steps, val_h=6):
    d = df_hist[["ds", col]].dropna().copy()
    s = d.set_index("ds")[col].astype(float)
    if len(s) < 6:
        fut_idx = pd.date_range(df_hist["ds"].max()+pd.offsets.MonthBegin(1), periods=steps, freq="MS")
        return pd.DataFrame({"ds": fut_idx, col: np.repeat(float(s.iloc[-1]) if len(s) else 0.0, steps)}), pd.DataFrame({"method":["last","snaive","ma3"],"weight":[1.0,0.0,0.0]})
    val_h = min(val_h, max(1, len(s)//4))
    train = s.iloc[:-val_h]
    val   = s.iloc[-val_h:]
    preds_val = {
        "snaive": seasonal_naive(train, val_h),
        "last":   last_value(train, val_h),
        "ma3":    moving_avg(train, val_h, 3),
    }
    w = inverse_mae_weights(val.values, preds_val)
    fut_idx = pd.date_range(s.index.max()+pd.offsets.MonthBegin(1), periods=steps, freq="MS")
    preds_fut = {
        "snaive": seasonal_naive(s, steps),
        "last":   last_value(s, steps),
        "ma3":    moving_avg(s, steps, 3),
    }
    mix = np.zeros(steps, dtype=float)
    for k, arr in preds_fut.items():
        mix += w.get(k,0.0) * arr
    out = pd.DataFrame({"ds": fut_idx, col: np.maximum(0.0, mix)})
    dbg = pd.DataFrame({"method": list(w.keys()), "weight": [w[k] for k in w]})
    return out, dbg

def time_series_splits(n, n_splits=3, min_train=24, test_h=3):
    if n < (min_train + n_splits*test_h):
        n_splits = max(1, (n - min_train)//max(1,test_h))
    if n_splits <= 0:
        n_splits = 1
    for i in range(n_splits):
        tr_end = min_train + i*test_h
        te_end = tr_end + test_h
        if te_end > n: break
        tr_idx = np.arange(0, tr_end)
        te_idx = np.arange(tr_end, te_end)
        yield tr_idx, te_idx

def fit_rf_with_rocv(X, y):
    grid = {
        "n_estimators": [300, 500],
        "max_depth": [None, 8, 12],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
    }
    best_params, best_mae = None, np.inf
    splits = list(time_series_splits(len(y), n_splits=3, min_train=min(24, max(12, len(y)//3)), test_h=3))
    if len(splits) == 0:
        splits = [(np.arange(0, max(1,len(y)-3)), np.arange(max(1,len(y)-3), len(y)))]
    for params in ParameterGrid(grid):
        maes = []
        for tr_idx, te_idx in splits:
            rf = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
            rf.fit(X[tr_idx], y[tr_idx])
            yhat = rf.predict(X[te_idx])
            maes.append(np.mean(np.abs(y[te_idx]-yhat)))
        m = float(np.mean(maes))
        if m < best_mae:
            best_mae = m
            best_params = params
    rf_final = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params).fit(X, y)
    return rf_final, best_params, best_mae

def smoothed_bootstrap_intervals(yhat_series, residuals, B=800, rng=None):
    rng = rng or np.random.default_rng(42)
    res = np.asarray(residuals, float); res = res[np.isfinite(res)]
    n = res.size
    mad = np.median(np.abs(res - np.median(res))) if n>0 else 0.0
    b_lap = (mad/np.sqrt(2)) if mad>0 else (np.std(res) if n>1 else 1.0)
    if n == 0:
        noise = rng.laplace(0.0, max(1e-3, b_lap), size=(len(yhat_series), B))
    else:
        idx = rng.integers(0, n, size=(len(yhat_series), B))
        base = res[idx]
        jitter = rng.laplace(0.0, max(0.25*b_lap, 1e-6), size=(len(yhat_series), B))
        noise = base + jitter
    sims = yhat_series.reshape(-1,1) + noise
    lo80 = np.nanquantile(sims, 0.10, axis=1)
    hi80 = np.nanquantile(sims, 0.90, axis=1)
    lo95 = np.nanquantile(sims, 0.025, axis=1)
    hi95 = np.nanquantile(sims, 0.975, axis=1)
    return sims, lo80, hi80, lo95, hi95

def stockout_stats(start_stock, sims):
    H, B = sims.shape
    cums = np.cumsum(np.maximum(0.0, sims), axis=0)
    crossed = (cums >= start_stock)
    p3 = float(np.mean(crossed[min(3,H)-1,:])) if H>=1 else 0.0
    p6 = float(np.mean(crossed[min(6,H)-1,:])) if H>=1 else 0.0
    first_cross = np.argmax(crossed, axis=0) + 1
    never = ~crossed.any(axis=0)
    first_cross[never] = H+1
    e_t = float(np.mean(first_cross))
    return p3, p6, e_t

def round_moq_lot(q, moq=0, lot=1):
    q = max(0.0, q)
    if lot <= 0: lot = 1
    if q > 0 and q < moq: q = moq
    q = math.ceil(q / lot) * lot
    return float(q)

panel = pd.read_csv(PANEL_PATH, parse_dates=["ds"]).sort_values(["sku","ds"]).reset_index(drop=True)
cfg   = pd.read_csv(CFG_PATH)
for col, default in {"T_CHECK":3,"H_COVER":6,"q_target":0.50,"lead_time_mo":0,"MOQ":0,"lot_size":1}.items():
    if col not in cfg.columns: cfg[col] = default

rows_summary = []
FEATS = ["orders","stock","orders_lag1","orders_lag3","stock_lag1","stock_lag3","y_lag1","y_lag3","orders_ratio","month","year"]
H_FUT = int(cfg["H_COVER"].max())

for sku, df in panel.groupby("sku", sort=False):
    row_cfg = cfg[cfg["sku"].astype(str)==str(sku)]
    if row_cfg.empty:
        T_CHECK=3; H_COVER=6; q_target=0.50; lead=0; moq=0; lot=1
    else:
        T_CHECK = int(row_cfg["T_CHECK"].iloc[0])
        H_COVER = int(row_cfg["H_COVER"].iloc[0])
        q_target= float(row_cfg["q_target"].iloc[0])
        lead    = int(row_cfg["lead_time_mo"].iloc[0])
        moq     = int(row_cfg["MOQ"].iloc[0])
        lot     = int(row_cfg["lot_size"].iloc[0])

    d = df[["ds","y","orders","stock"]].copy()
    d = ensure_ms_freq(d).fillna(0.0)
    feat = add_cal_and_lags(d)

    if len(feat) < 18:
        tr = feat.iloc[:-3]; va = feat.iloc[-3:]
    else:
        tr = feat.iloc[:-6]; va = feat.iloc[-6:]

    X_tr = tr[FEATS].to_numpy(); y_tr = tr["y"].to_numpy()
    rf, rf_params, rf_mae = fit_rf_with_rocv(X_tr, y_tr)

    fut_idx = pd.date_range(d["ds"].max()+pd.offsets.MonthBegin(1), periods=max(6, H_FUT+lead), freq="MS")
    exog_fut = pd.DataFrame({"ds": fut_idx})
    dbg_list = []
    for col in ["orders","stock"]:
        fc, dbg = exog_forecast_inverse_ensemble(d[["ds",col]], col, len(fut_idx), val_h=6)
        exog_fut = exog_fut.merge(fc, on="ds", how="left")
        dbg["exog"] = col; dbg_list.append(dbg)
    pd.concat(dbg_list, ignore_index=True).to_csv(OUT_DIR/f"debug/exog_weights_{sku}.csv", index=False)

    preds = []; full = feat.copy()
    for ds in fut_idx[:H_FUT]:
        full = full[~(full["ds"]==ds)]
        new_row = {"ds": ds, "y": 0.0,
                   "orders": float(exog_fut.loc[exog_fut["ds"]==ds,"orders"].values[0]),
                   "stock":  float(exog_fut.loc[exog_fut["ds"]==ds,"stock"].values[0])}
        full = pd.concat([full, pd.DataFrame([new_row])], ignore_index=True).sort_values("ds")
        full = add_cal_and_lags(full)
        X = full.loc[full["ds"]==ds, FEATS].to_numpy()
        yhat = float(rf.predict(X)[0])
        preds.append({"ds": ds, "yhat": max(0.0, yhat)})
        full.loc[full["ds"]==ds, "y"] = yhat

    pred_df = pd.DataFrame(preds)

    resids = []; rolling = tr.copy()
    for _, row in va.iterrows():
        mdl = RandomForestRegressor(**rf.get_params())
        mdl.fit(rolling[FEATS].to_numpy(), rolling["y"].to_numpy())
        ph = float(mdl.predict(row[FEATS].to_numpy().reshape(1,-1))[0])
        resids.append(row["y"] - ph)
        rolling = pd.concat([rolling, row.to_frame().T], ignore_index=True)
    resids = np.array(resids, dtype=float)

    sims, lo80, hi80, lo95, hi95 = smoothed_bootstrap_intervals(pred_df["yhat"].to_numpy(), resids, B=800)
    fc = pred_df.copy()
    fc["pi80_lo"] = np.maximum(0.0, lo80); fc["pi80_hi"] = np.maximum(0.0, hi80)
    fc["pi95_lo"] = np.maximum(0.0, lo95); fc["pi95_hi"] = np.maximum(0.0, hi95)
    fc.to_csv(OUT_DIR/f"forecasts/{sku}.csv", index=False, date_format="%Y-%m-%d")

    start_stock = float(d["stock"].iloc[-1]) if len(d)>0 else 0.0
    p3, p6, e_t = stockout_stats(start_stock, sims)

    horizon_cov = min(max(1, H_COVER), sims.shape[0])
    total_cov = sims[:horizon_cov,:].sum(axis=0)
    q_total = float(np.nanquantile(total_cov, q_target))
    raw_order = max(0.0, q_total - start_stock)
    order_qty = round_moq_lot(raw_order, moq=moq, lot=lot)

    rows_summary.append({
        "sku": sku,
        "rf_params": json.dumps(rf_params),
        "rf_mae_rocv": round(float(rf_mae), 3),
        "start_stock": round(start_stock, 3),
        "P_stockout_3m": round(p3, 5),
        "P_stockout_6m": round(p6, 5),
        "E_T_stockout_mo": round(e_t, 3),
        "T_CHECK": T_CHECK,
        "H_COVER": H_COVER,
        "q_target": q_target,
        "lead_time_mo": lead,
        "MOQ": moq,
        "lot_size": lot,
        "order_qty_recommended": round(order_qty, 2),
    })

summary = pd.DataFrame(rows_summary).sort_values("sku").reset_index(drop=True)
summary.to_csv(OUT_DIR/"metrics_summary.csv", index=False)
print("Done. Outputs:", str(OUT_DIR))


Done. Outputs: mnt/data/v7_per_sku_outputs
